In [45]:
import hashlib
import os
import platform
import uuid
import sys
import re
# 修改为自己的学号
STUDENT_ID = 20244080230
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]

RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号] {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID] {RUN_ID}")
print(f"[Python] {sys.version.split()[0]}")

[Conda 环境] exp01
[学号] 20244080230
[机器指纹] 2d80c6ac8fb85281
[RUN_ID] 20244080230-2d80c6ac8fb85281
[Python] 3.12.15


## 实验 1：正则表达式的基础匹配

**思路**

- 使用 `re.findall()` 进行匹配，会返回所有匹配到的子串列表；正则从左向右扫描文本，匹配成功就消耗对应字符。
- `\d+`：连续数字；`\s+`：连续空白符；`\b` 单词边界，用来精确匹配独立单词，避免匹配单词内片段。
- `|` 分支选择，**分支顺序会影响结果**，先写的分支优先匹配，匹配成功后不再尝试后面分支。
- 量词规则：`*` 0 次或多次、`+` 至少 1 次、`?` 0 或 1 次、`{n}` 精确 n 次、`{m,n}` m 到 n 次。

In [46]:
text1 = "cat on mat; later: cats and dogs"
print("原文本：", text1)

res_digit = re.findall(r"\d+", text1)
res_space = re.findall(r"\s+", text1)
res_cat_word = re.findall(r"\bcat\b", text1)

print(r"\d+ 匹配连续数字：", res_digit)
print(r"\s+ 匹配连续空白：", res_space)
print(r"\bcat\b 匹配独立单词cat：", res_cat_word)

原文本： cat on mat; later: cats and dogs
\d+ 匹配连续数字： []
\s+ 匹配连续空白： [' ', ' ', ' ', ' ', ' ', ' ']
\bcat\b 匹配独立单词cat： ['cat']


任务 1 说明：

- `\d+`：匹配一个或多个连续数字，文本无数字，结果为空
- `\s+`：匹配连续空格
- `\bcat\b`：单词边界，只匹配单独 cat，不会匹配 cats 里的 cat

In [47]:
text2 = "cat dog bird CAT"
print("原文本：", text2)
res_catdog = re.findall(r"cat|dog", text2)
print(r"cat|dog 匹配结果：", res_catdog)

res_a1 = re.findall(r"a|aa|aaa", "aaaa")
res_a2 = re.findall(r"aaa|aa|a", "aaaa")
print(r're.findall(r"a|aa|aaa", "aaaa") →', res_a1)
print(r're.findall(r"aaa|aa|a", "aaaa") →', res_a2)

原文本： cat dog bird CAT
cat|dog 匹配结果： ['cat', 'dog']
re.findall(r"a|aa|aaa", "aaaa") → ['a', 'a', 'a', 'a']
re.findall(r"aaa|aa|a", "aaaa") → ['aaa', 'a']


任务 2 说明：

- `|` 代表或，按书写顺序优先匹配；
- `a|aa|aaa` 先匹配短`a`，逐个消耗字符；
- `aaa|aa|a` 长串写前面，优先匹配更长字符串。

In [48]:
text3 = "ct cat caat caaat"
print("原文本：", text3)

res_star = re.findall(r"ca*t", text3)
res_plus = re.findall(r"ca+t", text3)
res_qmark = re.findall(r"ca?t", text3)
res_exact2 = re.findall(r"ca{2}t", text3)
res_range12 = re.findall(r"ca{1,2}t", text3)

print(r"ca*t  (*：0次或多次)：", res_star)
print(r"ca+t  (+：至少1次)：", res_plus)
print(r"ca?t  (?：0或1次)：", res_qmark)
print(r"ca{2}t ({2}：恰好2次)：", res_exact2)
print(r"ca{1,2}t ({1,2}：1~2次)：", res_range12)

原文本： ct cat caat caaat
ca*t  (*：0次或多次)： ['ct', 'cat', 'caat', 'caaat']
ca+t  (+：至少1次)： ['cat', 'caat', 'caaat']
ca?t  (?：0或1次)： ['ct', 'cat']
ca{2}t ({2}：恰好2次)： ['caat']
ca{1,2}t ({1,2}：1~2次)： ['cat', 'caat']


任务 3 说明：

- `ca*t`：a 可以出现 0 次及以上，ct、cat、caat 都满足
- `ca+t`：a 至少出现 1 次，ct 不满足
- `ca?t`：a 出现 0 次或 1 次，ct、cat 满足
- `ca{2}t`：a 严格 2 个，只匹配 caat
- `ca{1,2}t`：a 数量 1~2 个，cat、caat 满足

## 实验 2：分组、回溯引用与替换

**思路**

- 捕获组用 `()` 包裹，`re.findall()` 遇到捕获组时会按组返回元组，可分段提取用户名、域名等信息。
- 后向引用 `\1` 表示复用第 1 个捕获组匹配到的内容，用于查找重复出现的模式（如连续重复单词）。
- `re.sub()` 支持在替换字符串中通过 `\1 \2 \3` 引用捕获组，实现批量格式转换，且会保留原文的换行与其他字符。
- 匹配多种分隔符时用字符集 `[-/.]`，可一次性兼容横杠、斜杠、点号，适配不同格式的日期和电话号码。

In [49]:
text1 = "联系：alice@example.com；备选：bob@test.org"
# 第1组：用户名；第2组：域名
emails = re.findall(r"(\w+)@([\w.]+)", text1)

print("邮箱提取结果：")
for user, domain in emails:
    print(f"用户名：{user}，域名：{domain}")

邮箱提取结果：
用户名：alice，域名：example.com
用户名：bob，域名：test.org


任务 1 说明：

- 正则 `(\w+)@([\w.]+)` 用两个捕获组拆分邮箱结构，`@` 作为分隔锚点
- `re.findall()` 存在捕获组时，返回由各组内容组成的元组列表，而非完整匹配字符串

In [50]:
text2 = "cat cat, dog cat, bird bird"
# \1 引用第1个捕获组的匹配内容，匹配连续重复的单词
repeat_words = re.findall(r"\b(\w+)\s+\1\b", text2)

print("连续重复的单词：", repeat_words)
print("\n完整匹配片段：")
for m in re.finditer(r"\b(\w+)\s+\1\b", text2):
    print(m.group())

连续重复的单词： ['cat', 'bird']

完整匹配片段：
cat cat
bird bird


任务 2 说明：

- `\b(\w+)\b` 匹配一个完整单词并放入第 1 捕获组
- `\s+\1` 匹配空格后再次出现的相同单词，`\1` 就是对第 1 组的回溯引用
- 结果中 `cat cat` 和 `bird bird` 是连续重复项，`dog cat` 不重复因此不匹配

In [51]:
text3 = "2024-05-01\n2024/05/02\n2024.05.03"
print("转换前：")
print(text3)

# 依次捕获年、月、日，兼容 - / . 三种分隔符
result3 = re.sub(r"(\d{4})[-/.](\d{2})[-/.](\d{2})", r"\2-\3-\1", text3)

print("\n转换后：")
print(result3)

转换前：
2024-05-01
2024/05/02
2024.05.03

转换后：
05-01-2024
05-02-2024
05-03-2024


任务 3 说明：

- 三个捕获组分别对应 4 位年份、2 位月份、2 位日期
- 替换时按 `\2-\3-\1` 重排顺序，统一使用横杠分隔
- `[-/.]` 字符集同时匹配三种分隔符，替换后自动保留原文换行结构

In [52]:
text4 = "联系我：123-456-7890 或 987.654.3210"
print("转换前：", text4)

# 捕获三段号码，兼容 - 和 . 分隔符
result4 = re.sub(r"(\d{3})[-.](\d{3})[-.](\d{4})", r"(\1) \2-\3", text4)

print("转换后：", result4)

转换前： 联系我：123-456-7890 或 987.654.3210
转换后： 联系我：(123) 456-7890 或 (987) 654-3210


任务 4 说明：

- 三个捕获组依次匹配 3 位区号、3 位中间号、4 位尾号
- 替换为 `(区号) 中间号-尾号` 的标准格式，自动处理横杠和点号两种分隔
- 非号码的中文文本会原样保留，不影响其他内容

## 实验 3：使用 BeautifulSoup 解析 HTML

**思路**

- 直接解析本地 HTML 字符串，使用 Python 内置的 `html.parser` 作为解析器，无需发起网络请求。
- 单个标签可通过 `soup.标签名` 直接定位；`.name` 获取标签名称，`.string` 或 `.text` 获取标签内文本内容。
- `find_all()` 用于批量匹配元素：按 class 筛选需使用 `class_` 参数（避免与 Python 关键字冲突）；通过 `['href']` 读取标签的属性值。
- `get_text()` 会递归提取当前节点及所有子节点的纯文本，自动剥离 HTML 标签；调用节点的层级不同，返回的文本范围也不同。

In [53]:
from bs4 import BeautifulSoup

html_doc = """
<html><head><title>实验课程</title></head><body>
<h1>数据分析</h1>
<p class="course">第六讲：正则表达式</p>
<p class="course">第七讲：结构化数据</p>
<a href="https://example.org/lecture06">讲义六</a>
<a href="https://example.org/lecture07">讲义七</a>
</body></html>
"""

# 构建 HTML 解析树
soup = BeautifulSoup(html_doc, "html.parser")

# 提取页面标题文本
title_text = soup.title.string
# 提取 h1 的标签名与文本
h1_tag_name = soup.h1.name
h1_text = soup.h1.string

print("页面标题文本：", title_text)
print("h1 标签名：", h1_tag_name)
print("h1 文本内容：", h1_text)

页面标题文本： 实验课程
h1 标签名： h1
h1 文本内容： 数据分析


任务 1 说明：

- `BeautifulSoup(html_doc, "html.parser")` 将字符串解析为 DOM 树结构，后续所有查询都基于该对象
- `soup.title` 直接定位到页面中第一个 `<title>` 标签，`.string` 提取标签内纯文本
- `soup.h1.name` 返回标签本身的名称 `h1`，对应该元素的 HTML 标签类型

In [54]:
# 查找所有 class="course" 的段落
course_paras = soup.find_all("p", class_="course")
print("所有 class='course' 的段落：")
for p in course_paras:
    print("-", p.text)

print("\n所有链接信息：")
# 查找所有 a 标签
links = soup.find_all("a")
for link in links:
    link_text = link.get_text()
    link_href = link["href"]
    print(f"链接文字：{link_text}，链接地址：{link_href}")

所有 class='course' 的段落：
- 第六讲：正则表达式
- 第七讲：结构化数据

所有链接信息：
链接文字：讲义六，链接地址：https://example.org/lecture06
链接文字：讲义七，链接地址：https://example.org/lecture07


任务 2 说明：

- `class_` 参数用于匹配 HTML 的 class 属性，末尾加下划线是为了避开 Python 的 `class` 关键字
- `find_all("a")` 返回页面中所有 `<a>` 标签组成的列表，可遍历逐个处理
- 读取属性有两种常用写法：`标签['href']` 和 `标签.get('href')`，属性不存在时后者不会抛出异常


In [55]:
# 整个文档的可见文本
full_text = soup.get_text()
# 单个段落的文本
single_para_text = soup.p.get_text()
# 单个链接的文本
single_link_text = soup.a.get_text()

print("===== 整个文档文本 =====")
print(repr(full_text))  # repr 可显示换行、空格等空白字符原貌
print("\n===== 单个段落文本 =====")
print(single_para_text)
print("\n===== 单个链接文本 =====")
print(single_link_text)

===== 整个文档文本 =====
'\n实验课程\n数据分析\n第六讲：正则表达式\n第七讲：结构化数据\n讲义六\n讲义七\n\n'

===== 单个段落文本 =====
第六讲：正则表达式

===== 单个链接文本 =====
讲义六


任务 3 说明：

- 对整个文档调用 `get_text()` 会提取所有标签内的文字，保留换行结构，但完全剥离 HTML 标签
- 对单个段落、单个链接调用时，仅提取对应节点及其子节点的文本，覆盖范围更小
- 实际使用中可加参数 `get_text(strip=True)`，自动去除文本首尾多余的空白和换行

## 实验 3 结论

1. **HTML 解析与基础定位**：使用 `BeautifulSoup` 配合 Python 内置的 `html.parser`，可将字符串形式的 HTML 解析为结构化的 DOM 树；通过 `soup.标签名` 可直接定位页面中首个对应元素，`.name` 可获取标签名称，`.string` / `.text` 可提取标签内的文本内容。
2. **批量元素查找与属性读取**：`find_all()` 可批量匹配指定标签，配合 `class_` 参数可按 CSS 类名筛选目标元素；通过 `标签['属性名']` 的语法可读取 href 等 HTML 属性，实现链接地址、类名等信息的批量提取。
3. **文本提取的范围差异**：`get_text()` 会递归提取当前节点及其所有子节点的纯文本，自动剥离全部 HTML 标签；调用节点的层级决定了文本提取的范围 —— 对文档根节点调用会提取整页可见文本，对单个标签调用仅提取对应局部内容；可通过 `strip=True`、`separator` 参数进一步优化空白与换行的输出格式。